# Lumpyspace: Live Training Monitor
This notebook provides real-time visualization of the PINN training progress by reading the live CSV logs generated by `src/training/run.py`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time
import os

# Path to the production logs
log_path = "../logs/training_metrics.csv"

def monitor_convergence():
    while True:
        if not os.path.exists(log_path):
            clear_output(wait=True)
            print(f"Waiting for log file: {log_path}...")
            time.sleep(5)
            continue
            
        try:
            df = pd.read_csv(log_path)
            if len(df) > 1:
                clear_output(wait=True)
                fig, ax = plt.subplots(1, 2, figsize=(15, 5))
                
                # Plot Total Loss (Log Scale)
                ax[0].plot(df['step'], df['loss'].astype(float), label='Total Loss', color='black', linewidth=2)
                ax[0].set_yscale('log')
                ax[0].set_xlabel('Training Step')
                ax[0].set_ylabel('Loss (Total)')
                ax[0].set_title('Global Convergence Profile')
                ax[0].grid(True, which="both", ls="-", alpha=0.3)
                ax[0].legend()
                
                # Plot Components
                ax[1].plot(df['step'], df['l_phys'].astype(float), label='Physics (EFE Residual)', alpha=0.8)
                ax[1].plot(df['step'], df['l_data'].astype(float), label='Data (SN Chi-Squared)', alpha=0.8)
                ax[1].set_yscale('log')
                ax[1].set_xlabel('Training Step')
                ax[1].set_ylabel('Loss Component')
                ax[1].set_title('Residual Breakdown')
                ax[1].grid(True, which="both", ls="-", alpha=0.3)
                ax[1].legend()
                
                plt.tight_layout()
                plt.show()
                
                print(f"Last Update: {time.strftime('%H:%M:%S')} | Steps: {df['step'].iloc[-1]} | Current Loss: {df['loss'].iloc[-1]:.4e}")
        except Exception as e:
            print(f"Error reading logs: {e}")
        
        time.sleep(10) # Refresh every 10 seconds to reduce IO overhead

monitor_convergence()

: 